# Pipeline

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

housing = pd.read_csv('/home/siyam/MyCodes/MachineLearning/Book/datasets/housing/housing.csv')

housing['income_cat'] = pd.cut(
    housing['median_income'],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1,2,3,4,5,]
)

X = housing.drop(['median_house_value', 'income_cat'], axis=1)
y = housing['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=housing['income_cat']
)


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns

num_pipeline = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler' , StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessing = ColumnTransformer(
    [
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

pipeline = Pipeline(
    [
        ('preprocessing', preprocessing),
        ('model', LinearRegression())
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

j = 0
while j < 10:
    print(f"{y_test.iloc[j]}  <======>  {y_pred[j]:.2f}")
    j += 1


from sklearn.metrics import r2_score
print(f'R SQUARED: {r2_score(y_test, y_pred):.2f}')

from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_test, y_pred)
print(f'RMSE: {rmse:.2f}')

print("Train R²:", pipeline.score(X_train, y_train))
print("Test R² :", pipeline.score(X_test, y_test))

397700.0  <======>  383993.92
202900.0  <======>  252310.54
310000.0  <======>  372503.60
314300.0  <======>  279372.70
187500.0  <======>  225899.33
141700.0  <======>  212940.69
104900.0  <======>  163517.46
275100.0  <======>  346138.25
167000.0  <======>  257846.47
457700.0  <======>  258190.50
R SQUARED: 0.62
RMSE: 71002.84
Train R²: 0.6498543749328045
Test R² : 0.6233813507638213


In [20]:
print(housing.info())

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           20640 non-null  float64 
 1   latitude            20640 non-null  float64 
 2   housing_median_age  20640 non-null  float64 
 3   total_rooms         20640 non-null  float64 
 4   total_bedrooms      20433 non-null  float64 
 5   population          20640 non-null  float64 
 6   households          20640 non-null  float64 
 7   median_income       20640 non-null  float64 
 8   median_house_value  20640 non-null  float64 
 9   ocean_proximity     20640 non-null  str     
 10  income_cat          20640 non-null  category
dtypes: category(1), float64(9), str(1)
memory usage: 1.6 MB
None


In [3]:
housing_cat = housing[['ocean_proximity']]

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
housing_cat_encoded = ohe.fit_transform(housing_cat)

print(ohe.categories_)
print(housing_cat_encoded[100:111])

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
      dtype=object)]
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (11, 5)>
  Coords	Values
  (0, 0)	1.0
  (1, 1)	1.0
  (2, 4)	1.0
  (3, 0)	1.0
  (4, 0)	1.0
  (5, 0)	1.0
  (6, 4)	1.0
  (7, 0)	1.0
  (8, 0)	1.0
  (9, 0)	1.0
  (10, 1)	1.0
